# Phase 1 — Data Ingestion
Rainfall-triggered landslide detection pipeline

Scope: pull every raw source needed at the right native resolution, with nothing silently missing. **No cleaning, masking, or alignment happens in this notebook** — that is Phase 2. Each source below is independently testable on a small window before scaling up.

## Setup

In [ ]:
import ee
import geemap.core as geemap
import pandas as pd
import numpy as np

# Use a dedicated Cloud project for reproducibility across sessions
ee.Authenticate()
ee.Initialize(project='gen-lang-client-0218474328')  # <-- replace with your GEE-linked Cloud project

print("Earth Engine initialized:", ee.data.getAssetRoots() is not None)


## Region of interest
Himalayan belt bounding box. Kept identical to prior work so downstream comparisons stay valid.

In [ ]:
lat_min, lat_max = 26.0, 35.0
lon_min, lon_max = 74.0, 97.0

roi = ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])

print(f"ROI area: {roi.area().divide(1e6).getInfo():,.0f} km^2")


## 1. Sentinel-2 SR (optical) + s2cloudless probability
Pulled raw and joined to its cloud-probability collection here so the pair travels together into Phase 2 — masking itself happens there, not now.

In [ ]:
S2_START, S2_END = "2017-01-01", "2023-12-31"

s2_sr = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(roi)
         .filterDate(S2_START, S2_END))

s2_cloud_prob = (ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
                  .filterBounds(roi)
                  .filterDate(S2_START, S2_END))

# Join SR imagery to its per-pixel cloud probability by the shared system:index
s2_join = ee.Join.saveFirst('s2cloudless').apply(
    primary=s2_sr,
    secondary=s2_cloud_prob,
    condition=ee.Filter.equals(leftField='system:index', rightField='system:index')
)
s2_with_prob = ee.ImageCollection(s2_join)

print("Sentinel-2 SR scenes in ROI/date range:", s2_sr.size().getInfo())
print("Joined (SR + cloud probability) scenes:", s2_with_prob.size().getInfo())
print("SR bands:", s2_sr.first().bandNames().getInfo())


## 2. Sentinel-1 GRD (SAR)
All-weather signal — this is what actually covers monsoon-season events when optical imagery is cloud-blocked.

In [ ]:
S1_START, S1_END = "2017-01-01", "2023-12-31"

s1_grd = (ee.ImageCollection("COPERNICUS/S1_GRD")
          .filterBounds(roi)
          .filterDate(S1_START, S1_END)
          .filter(ee.Filter.eq("instrumentMode", "IW"))
          .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
          .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
          .select(["VV", "VH"]))

print("Sentinel-1 IW VV/VH scenes in ROI/date range:", s1_grd.size().getInfo())
print("Bands:", s1_grd.first().bandNames().getInfo())


## 3. Rainfall — CHIRPS daily
Native resolution ~5.5 km — deliberately *not* resampled here. Extraction scale is decided per-source in Phase 3, not baked in now.

In [ ]:
chirps = (ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
          .filterBounds(roi)
          .filterDate("2014-01-01", "2023-12-31"))

print("CHIRPS daily images in ROI/date range:", chirps.size().getInfo())
print("Bands:", chirps.first().bandNames().getInfo())
print("Native resolution: ~5566 m (0.05 deg)")


## 4. DEM — SRTM (primary) + ALOS World 3D / AW3D30 (cross-check)
SRTM stays the primary 30 m reference grid for Phase 5. AW3D30 is pulled alongside it for a high-relief accuracy comparison, not as a replacement.

In [ ]:
srtm = ee.Image("USGS/SRTMGL1_003").select("elevation").clip(roi)
aw3d30 = ee.ImageCollection("JAXA/ALOS/AW3D30/V3_2").select("DSM").mosaic().clip(roi)

srtm_slope = ee.Terrain.slope(srtm)
srtm_aspect = ee.Terrain.aspect(srtm)

print("SRTM band:", srtm.bandNames().getInfo())
print("AW3D30 band:", aw3d30.bandNames().getInfo())

# Cheap sanity check: mean elevation difference at a coarse scale over the ROI
diff = srtm.subtract(aw3d30.rename("elevation"))
mean_diff = diff.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=roi, scale=1000, maxPixels=1e9, bestEffort=True
).getInfo()
print("Mean SRTM - AW3D30 elevation difference (m, coarse check):", mean_diff)


## 5. Landslide inventory — raw load only
Source: NASA COOLR Landslide Viewer export (merges the curated Global Landslide Catalog with other named regional inventories and citizen reports — see `Imported Event Source Catalog` below).

No filtering, deduplication, or accuracy screening yet — that is Phase 2. This step just confirms the file is readable and inspects the fields Phase 2's cleaning logic will actually use: positional accuracy, trigger type (to isolate rainfall events), and source catalog.

In [ ]:
# Upload Landslide_Reports.csv to the Colab session (or mount Drive) before running this cell.
df_raw = pd.read_csv("Landslide_Reports.csv")

print(f"Raw inventory rows: {len(df_raw)}")
print(f"Columns: {list(df_raw.columns)}")

# Event Date is stored as 'DD/MM/YYYY, HH:MM' (with some rows blank)
parsed_dates = pd.to_datetime(df_raw["Event Date"], format="%d/%m/%Y, %H:%M", errors="coerce")
df_raw["event_date_parsed"] = parsed_dates
print(f"\nParseable dates: {parsed_dates.notna().sum()} / {len(df_raw)}")
print(f"Date range: {parsed_dates.min()} to {parsed_dates.max()}")

in_roi = df_raw[
    (df_raw["Latitude"].between(lat_min, lat_max)) &
    (df_raw["Longitude"].between(lon_min, lon_max))
]
print(f"\nRows falling inside the ROI bounding box (unfiltered by date/quality/trigger): {len(in_roi)}")

print("\n--- Location Accuracy (full dataset) ---")
print(df_raw["Location Accuracy"].value_counts(dropna=False))

print("\n--- Landslide Trigger (full dataset) ---")
print(df_raw["Landslide Trigger"].value_counts(dropna=False))

print("\n--- Imported Event Source Catalog (full dataset) ---")
print(df_raw["Imported Event Source Catalog"].value_counts(dropna=False))

# Rainfall-relevant trigger labels present in this export — Phase 2 will filter to these,
# consistent with the rainfall-only scope decision.
rainfall_triggers = ["Downpour", "Rain", "Continuous rain", "Monsoon", "Tropical cyclone"]
roi_rainfall = in_roi[in_roi["Landslide Trigger"].isin(rainfall_triggers)]
print(f"\nROI rows with a rainfall-related trigger label (preview of Phase 2's filter): {len(roi_rainfall)}")


## Quick visual verification
One map with a layer from each source, purely to confirm every pull is actually returning data over the ROI before Phase 2 begins.

In [ ]:
m = geemap.Map(center=[30.5, 85.5], zoom=5)

m.addLayer(roi, {"color": "gray"}, "ROI")
m.addLayer(srtm, {"min": 0, "max": 7000, "palette": ["blue", "green", "yellow", "brown", "white"]}, "SRTM elevation")
m.addLayer(s1_grd.median(), {"bands": ["VV"], "min": -25, "max": 0}, "Sentinel-1 VV (median)")
m.addLayer(s2_sr.median(), {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}, "Sentinel-2 RGB (median)")
m.addLayer(chirps.select("precipitation").mean(), {"min": 0, "max": 20, "palette": ["white", "blue"]}, "CHIRPS mean daily rainfall")

m


## Ingestion summary — data dictionary (Phase 1 deliverable)
Documented now so Phase 2/3 don't have to rediscover native resolution and coverage by trial and error.

In [ ]:
ingestion_summary = pd.DataFrame([
    {"source": "Sentinel-2 SR + s2cloudless", "collection": "COPERNICUS/S2_SR_HARMONIZED", "native_res_m": 10, "revisit_days": "~5", "date_range": f"{S2_START} to {S2_END}"},
    {"source": "Sentinel-1 GRD (IW, VV/VH)", "collection": "COPERNICUS/S1_GRD", "native_res_m": 10, "revisit_days": "6-12", "date_range": f"{S1_START} to {S1_END}"},
    {"source": "CHIRPS daily rainfall", "collection": "UCSB-CHG/CHIRPS/DAILY", "native_res_m": 5566, "revisit_days": "1", "date_range": "2014-01-01 to 2023-12-31"},
    {"source": "SRTM DEM (primary)", "collection": "USGS/SRTMGL1_003", "native_res_m": 30, "revisit_days": "static", "date_range": "n/a"},
    {"source": "AW3D30 DEM (cross-check)", "collection": "JAXA/ALOS/AW3D30/V3_2", "native_res_m": 30, "revisit_days": "static", "date_range": "n/a"},
    {"source": "Landslide inventory (raw, NASA COOLR)", "collection": "Landslide_Reports.csv (external)", "native_res_m": "point", "revisit_days": "n/a", "date_range": "see parsed range above"},
])

ingestion_summary


---
**Next: Phase 2 — preprocessing.** Cloud/shadow masking (s2cloudless + SCL), spatial/temporal alignment, inventory cleaning, and the joint spatio-temporal negative sampling logic all build on the raw collections pulled above.